In [ ]:
import os
import polars as pl
os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import *
from jetutils.data import *
from jetutils.geospatial import *
import altair as alt
import matplotlib.pyplot as plt
alt.data_transformers.enable("vegafusion")

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

In [ ]:
jets = pl.read_parquet(f"{DATADIR}/exp8/t2mmeters_relative.parquet")

In [ ]:
polars_to_xarray(jets.filter(pl.col("time") == pl.col("time").first(), ~pl.col("is_polar")).group_by("is_polar", "norm_index", "n", maintain_order=True).agg(pl.col("t2m_interp").mean()), ["is_polar", "norm_index", "n"])[0].T.plot()

# jet tracking tests

In [ ]:
all_jets_one_df = pl.read_parquet(f"{DATADIR}/exp8/all_jets_one_df.parquet")
phat_jets = all_jets_one_df.filter((pl.col("is_polar").mean().over(["time", "jet ID"]) < 0.5) | ((pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5) & (pl.col("int").mode().first().over(["time", "jet ID"]) > 1.3e8)))
phat_jets_catd = phat_jets.with_columns(**{"jet ID": (pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5).cast(pl.UInt32())})

cross_new_ofile = Path(DATADIR, "exp8/cross_new.parquet")
if cross_new_ofile.is_file():
    cross_new = pl.read_parquet(cross_new_ofile)
else:
    cross_new = track_jets(phat_jets)
    cross_new.write_parquet(cross_new_ofile)
    

cross_new_catd_ofile = Path(DATADIR, "exp8/cross_new_catd.parquet")
if cross_new_catd_ofile.is_file():
    cross_catd_new = pl.read_parquet(cross_new_catd_ofile)
else:
    cross_catd_new = track_jets(phat_jets_catd)
    cross_catd_new.write_parquet(cross_new_catd_ofile)

In [ ]:
cross_new_, summary_comp = connected_from_cross(all_jets_one_df=all_jets_one_df, cross=cross_new, dist_thresh=4e6, overlap_thresh=0.6, dis_polar_thresh=0.2)


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from itertools import pairwise

spell = np.random.choice(summary_comp.filter(pl.col("len") > 8)["spell"].unique())
this_spell = summary_comp.filter(pl.col("spell") == spell)[:-1]
segments = np.array([[[i, b], [i + 1, e]] for i, (b, e) in enumerate(pairwise(this_spell["is_polar"]))])
cmap = colormaps.hawaii
norm = Normalize(vmin=0, vmax=4e6)
colors = cmap(norm(this_spell["dist"]))

fig, ax = plt.subplots()
ax.scatter(np.arange(this_spell.shape[0]), this_spell["is_polar"])
lines = LineCollection(segments, colors=colors)
ax.add_collection(lines)
fig.colorbar(ScalarMappable(norm, cmap), ax=ax)

In [ ]:
spells_list = spells_from_cross(all_jets_one_df=all_jets_one_df, cross=cross_new, dist_thresh=2e6, overlap_thresh=0.6, dis_polar_thresh=0.6, season=summer)

# many different measures

In [ ]:
from similaritymeasures import frechet_dist, curve_length_measure, area_between_two_curves, dtw, pcm

dists = {f.__name__: f for f in [frechet_dist, curve_length_measure, area_between_two_curves, dtw, pcm]}

ncol, nrow = 4, 4
n = ncol * nrow
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes = axes.ravel()
times = all_jets_one_df["time"].unique()
already_picked_index = []

for i, ax in enumerate(axes):
    valid = False
    while not valid:
        possible_times = np.arange(len(times) - 1)
        possible_times = np.delete(possible_times, already_picked_index)
        time_index = np.random.choice(possible_times)
        already_picked_index.append(time_index)
        t = times[time_index.item()]
        onejet = all_jets_one_df.filter(pl.col("time") == t, pl.col("jet ID") == 0)
        nextjet = all_jets_one_df.filter(pl.col("time") == t + datetime.timedelta(hours=6), pl.col("jet ID") == 0)
        if len(onejet) > 0 and len(nextjet) > 0:
            valid = True
    
    ax.scatter(onejet["lon"], onejet["lat"], c=onejet["index"], cmap=colormaps.speed)
    ax.scatter(nextjet["lon"], nextjet["lat"], c=nextjet["index"], cmap=colormaps.amp)

    dists_results = {}
    for dname, d in dists.items():
        dists_results[dname] = d(onejet["lon", "lat"].to_numpy(), nextjet["lon", "lat"].to_numpy())
        if isinstance(dists_results[dname], tuple):
            dists_results[dname] = dists_results[dname][0]
    for k, (name, res) in enumerate(dists_results.items()):
        ax.annotate(f"{name[0]}: {res:.1f}", (0.01, 0.95 - 0.06 * k), xycoords="axes fraction", fontsize=11)

# jet detection tests

In [ ]:
from jetutils.jet_finding import find_all_jets

keys = ["250", "high", "2pvu_reduced"]
dss = {}
ds_prepd = {}
all_jets = {}
smooth_s = 7
n_coarsen = 3
base_s_thresh = 0.6
for key in keys:
    dss[key] = standardize(xr.open_dataset(f"{DATADIR}/ERA5/sample_{key}.nc")).sel(lat=slice(15, 80), lon=slice(-80, 40)).load()
    if "s" not in dss[key]:
        dss[key]["s"] = np.hypot(dss[key]["u"], dss[key]["v"])
    all_jets[key] = find_all_jets(dss[key], smooth_s=smooth_s, n_coarsen=n_coarsen, base_s_thresh=base_s_thresh, alignment_thresh=0.7, hole_size=4)
    ds_prepd[key] = preprocess_ds(dss[key], smooth_s=smooth_s, n_coarsen=n_coarsen)

In [ ]:
def compute_laplacian(ds: xr.Dataset):
    lon, lat = ds["lon"].values, ds["lat"].values
    xlon, ylat = np.meshgrid(lon, lat)

    dlaty, dlatx = np.gradient(ylat)
    dlony, dlonx = np.gradient(xlon)

    dy = RADIUS * np.radians(dlaty)
    dx = RADIUS * np.radians(dlaty) * degcos(ylat)
    
    axis_y = np.where(np.array(ds["s"].dims) == "lat")[0].item()
    axis_x = np.where(np.array(ds["s"].dims) == "lon")[0].item()
    
    u = ds["u"]
    v = ds["v"]
    s = np.hypot(u, v)
    
    dudx = u.copy(data=np.gradient(u, axis=axis_x)) / dx[None, :, :]
    dvdy = v.copy(data=np.gradient(v, axis=axis_y)) / dy[None, :, :]
    
    d2udx2 = dudx.copy(data=np.gradient(dudx, axis=axis_y)) / dx[None, :, :]
    d2vdy2 = dvdy.copy(data=np.gradient(dvdy, axis=axis_x)) / dy[None, :, :]
    
    return d2udx2 + d2vdy2

def compute_tangential_derivative(ds: xr.Dataset, of: str = "alpha"):
    lon, lat = ds["lon"].values, ds["lat"].values
    xlon, ylat = np.meshgrid(lon, lat)

    dlaty, dlatx = np.gradient(ylat)
    dlony, dlonx = np.gradient(xlon)

    dy = RADIUS * np.radians(dlaty)
    dx = RADIUS * np.radians(dlaty) * degcos(ylat)
    
    axis_y = np.where(np.array(ds["s"].dims) == "lat")[0].item()
    axis_x = np.where(np.array(ds["s"].dims) == "lon")[0].item()
    
    u = ds["u"]
    v = ds["v"]
    if "alpha" not in ds:
        ds["alpha"] = np.atan2(v, u)
    da = ds[of]
    
    ddadx = da.copy(data=np.gradient(da, axis=axis_x)) / dx[None, :, :]
    ddady = da.copy(data=np.gradient(da, axis=axis_y)) / dy[None, :, :]
    
    return ddadx * np.cos(ds["alpha"]) + ddady * np.sin(ds["alpha"])

In [ ]:
ds = dss["250"]
ds["alpha"] = np.atan2(ds["v"], ds["u"])
curvature = compute_tangential_derivative(ds, "alpha") * 1e6

In [ ]:
(curvature[0] * ds["s"][0]).plot(vmin = -100, vmax=100, cmap=colormaps.BlWhRe)
ds.isel(time=0, lon=slice(None, None, 5), lat=slice(None, None, 5)).plot.quiver(x="lon", y="lat", u="u", v="v", color="white")

In [ ]:
compute_curvature(dss["250"].isel(time=[0])).plot()
dss["250"].isel(time=0, lon=slice(None, None, 3), lat=slice(None, None, 3)).plot.quiver(x="lon", y="lat", u="u", v="v")

In [ ]:
def compute_rank(da):
    ranks = da.copy(data=np.argsort(np.argsort((da.values.ravel()))).reshape(da.shape))
    return ranks / ranks.max()

all_times = dss["250"].time[dss["250"].time.dt.season=="JJA"]
n_col = 3
n_row = 5
times = np.random.choice(all_times, size=n_row)
clu = Clusterplot(ncol=n_col, nrow=n_row, region=get_region(dss["250"]))
titles = [f"{key.rstrip("_reduced")}, {np.datetime_as_string(time, unit='D')}" for time in times for key in keys]
to_plot = [dss[key].sel(time=time)["theta"]for time in times for key in keys]
_ = clu.add_contourf(to_plot, titles=titles, cmap=colormaps.amp, levels=np.arange(300, 410, 10).tolist())
# to_plot = [compute_rank(dss[key].sel(time=time)["s"]) for time in times for key in keys]
# _ = clu.add_contour(to_plot, levels=[base_s_thresh], colors=["green"])
cmap_2 = colormaps.lapaz_r
cmap_2.set_under("green")
for i, key in enumerate(keys):
    for j, time in enumerate(times):
        k = n_col * j + i
        ax = clu.axes[k]
        jets_ = all_jets[key].filter(pl.col("time") == time)
        for _, jet_ in jets_.group_by("jet ID"):
            lo, la, s, theta, alignment, u, v, sigma_theta, theta_orig = jet_["lon", "lat", "s", "theta", "alignment", "u", "v", "sigma_theta", "theta_orig"]
            # ax.scatter(lo - clu.central_longitude, la, s=(s - 10) ** 2 / 100, c=(sigma_theta * RADIUS), cmap=colormaps.BlWhRe, vmin=-400, vmax=400)
            ax.scatter(lo - clu.central_longitude, la, s=(s - 10) ** 2 / 100, c=alignment, cmap=cmap_2, vmin=0.5, vmax=1)
            # ax.scatter(lo - clu.central_longitude, la, s=(s - 10) ** 2 / 100, c=theta_orig, cmap=colormaps.cet_l_bmw, vmin=300, vmax=400)

In [ ]:
these = all_jets["250"].group_by("jet ID", "time").agg(haversine(pl.col("lon").first(), pl.col("lat").first(), pl.col("lon").last(), pl.col("lat").last())).filter(pl.col("literal") < 1.3e6).join(all_jets["250"], on=["jet ID", "time"])

for _, this_one in these.group_by("time", "jet ID"):
    a, b = this_one["lon", "lat"]
    plt.plot(a, b)
    if np.random.randint(0, 10) == 8:
        break

In [ ]:
distances = all_jets["250"].group_by("jet ID", "time").agg(haversine(pl.col("lon").first(), pl.col("lat").first(), pl.col("lon").last(), pl.col("lat").first()).replace([-float("inf"), float("inf"), 0], 1e3))
_ = plt.hist(distances["literal"].log10(), bins=50)

In [ ]:
(sigma_theta * RADIUS).max()

In [ ]:
def compute_alignment_2(
    all_contours: DataFrame, periodic: bool = False
) -> DataFrame:
    """
    This function computes the alignment criterion for zero-sigma-contours. It is the scalar product betweeen the vector from a contour point to the next and the horizontal wind speed vector.
    """
    index_columns = get_index_columns(
        all_contours, ("member", "time", "cluster", "spell", "relative_index")
    )
    dlon = diff_maybe_periodic("lon", periodic)
    dlat = central_diff("lat")
    ds = haversine_from_dl(pl.col("lat"), dlon, dlat)
    align_x = pl.col("u") / pl.col("s") * haversine_from_dl(pl.col("lat"), dlon, pl.lit(0)) / ds
    align_y = pl.col("v") / pl.col("s") * haversine_from_dl(pl.col("lat"), pl.lit(0), dlat) / ds
    alignment = align_x + align_y
    return all_contours.with_columns(
        alignment=alignment.over([*index_columns, "contour"])
    )

In [ ]:
compute_alignment_2(all_jets["250"], False)

In [ ]:
jet_["alignment"]

In [ ]:
ds = dss["2pvu_reduced"]
n_coarsen = 3
smooth_s = 5
# ds = coarsen_da(ds, n_coarsen=n_coarsen)
# if "s" not in ds:
#     ds["s"] = np.sqrt(ds["u"] ** 2 + ds["v"] ** 2)

# if smooth_s is not None:
#     smooth_map = ("win", smooth_s)
#     smooth_map = {"lon": smooth_map, "lat": smooth_map}
#     ds = ds.rename({var: f"{var}_orig" for var in ["u", "v", "s"]})
#     for var in ["u", "v", "s"]:
#         to_smooth = ds[f"{var}_orig"]
#         ds[var] = to_smooth.copy(data=smooth(to_smooth, smooth_map=smooth_map))
ds = ds.assign_coords(
    {
        "x": np.radians(ds["lon"]) * RADIUS,
        "y": RADIUS
        * np.log(
            (1 + np.sin(np.radians(ds["lat"])) / np.cos(np.radians(ds["lat"])))
        ),
    }
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    ds["sigma"] = (
        -ds["u"] * ds["s"].differentiate("y") + ds["v"] * ds["s"].differentiate("x")
    ) / ds["s"]
# fft_smoothing = 1.0 if ds["sigma"].min() < -0.0001 else 0.8
sigma1 = ds["sigma"]

In [ ]:
def compute_norm_derivative(ds: xr.Dataset, of: str = "s"):
    lon, lat = ds["lon"].values, ds["lat"].values
    xlon, ylat = np.meshgrid(lon, lat)

    dlaty, dlatx = np.gradient(ylat)
    dlony, dlonx = np.gradient(xlon)

    dy = RADIUS * np.radians(dlaty)
    dx = RADIUS * np.radians(dlaty) * degcos(ylat)
    
    axis_y = np.where(np.array(ds["s"].dims) == "lat")[0].item()
    axis_x = np.where(np.array(ds["s"].dims) == "lon")[0].item()
    
    u = ds["u"]
    v = ds["v"]
    s = np.hypot(u, v)
    da = ds[of]
    
    ddady = da.copy(data=np.gradient(da, axis=axis_y)) / dy[None, :, :]
    ddadx = da.copy(data=np.gradient(da, axis=axis_x)) / dx[None, :, :]
    
    return (- u * ddady + v * ddadx) / s

sigma2 = compute_norm_derivative(dss["2pvu_reduced"])